# torchdata smoke test

In [ ]:
# torchdata.nodes composable dataloading: IterableWrapper -> Mapper -> Batcher -> Loader.
# Runs the real node runtime (reset/next protocol + Loader iteration), not just imports.
from torchdata.nodes import IterableWrapper, Mapper, Batcher, Loader

source = IterableWrapper(range(20))
squared = Mapper(source, map_fn=lambda x: x * x)
batched = Batcher(squared, batch_size=4, drop_last=True)
loader = Loader(batched)

batches = list(loader)
assert len(batches) == 5, batches
assert batches[0] == [0, 1, 4, 9], batches[0]
# Loader restarts cleanly: a second full pass yields identical data.
assert list(loader) == batches
print(f"nodes pipeline produced {len(batches)} batches; first = {batches[0]}")

In [ ]:
# ParallelMapper spawns a real worker pool (method='thread'); Prefetcher runs a
# background prefetch thread. in_order=True must preserve input order.
import gc, time
from torchdata.nodes import IterableWrapper, ParallelMapper, Prefetcher, Loader

def slow_double(x):
    time.sleep(0.001)  # force overlap across the worker threads
    return x * 2

pm = ParallelMapper(IterableWrapper(range(16)), map_fn=slow_double,
                    num_workers=4, in_order=True, method="thread")
pipeline = Prefetcher(pm, prefetch_factor=2)
loader = Loader(pipeline)
out = list(loader)
assert out == [x * 2 for x in range(16)], out
print(f"ParallelMapper(4 threads)+Prefetcher -> {len(out)} items, ordered: {out[:4]}...")
# Deterministic worker-pool teardown while the interpreter is alive (avoids shutdown-race warnings).
del loader, pipeline, pm
gc.collect()

In [ ]:
# nodes checkpointing: Loader.state_dict() captures mid-stream position;
# load_state_dict() on a fresh identical pipeline resumes exactly where it left off.
from torchdata.nodes import IterableWrapper, Mapper, Loader

def build():
    return Loader(Mapper(IterableWrapper(range(10)), map_fn=lambda x: x + 1))

loader = build()
it = iter(loader)
seen = [next(it) for _ in range(3)]        # consume 3 of 10 items
ckpt = loader.state_dict()

resumed_loader = build()
resumed_loader.load_state_dict(ckpt)
tail = list(resumed_loader)                 # resume from the checkpoint

assert seen + tail == list(build()), (seen, tail)
print(f"nodes resume round-trip OK: consumed {seen}, resumed {tail}")

In [ ]:
# StatefulDataLoader is a drop-in torch.utils.data.DataLoader replacement; its CPU
# surface (construction, iteration, state_dict/load_state_dict) is covered by the
# mid-epoch-resume cell below, so THIS cell owns only the device half: pin_memory
# and the .cuda() transfer. num_workers=0 sidesteps the Python 3.14 notebook
# forkserver-reimport pitfall.
import torch
from torch.utils.data import TensorDataset
from torchdata.stateful_dataloader import StatefulDataLoader

assert torch.cuda.is_available(), (
    f"requires-gpu cell ran but torch.cuda.is_available() is False — torch {torch.__version__}, "
    f"built against CUDA {torch.version.cuda}, device_count={torch.cuda.device_count()}"
)

torch.manual_seed(0)
ds = TensorDataset(torch.arange(40, dtype=torch.float32).reshape(40, 1), torch.arange(40))
dl = StatefulDataLoader(ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
xb, yb = next(iter(dl))
assert xb.shape == (8, 1) and yb.tolist() == list(range(8))
assert xb.is_pinned(), "pin_memory=True must hand back pinned host memory"
xb = xb.cuda()
assert xb.is_cuda
print(f"GPU batch on {xb.device}, shape {tuple(xb.shape)}")

In [ ]:
# Headline torchdata feature: exact mid-epoch resume. Iterate part-way, checkpoint,
# then resume on a fresh loader and verify byte-for-byte continuity of the sample stream.
import torch
from torch.utils.data import TensorDataset
from torchdata.stateful_dataloader import StatefulDataLoader

ds = TensorDataset(torch.arange(24).reshape(24, 1))
def make():
    return StatefulDataLoader(ds, batch_size=4, shuffle=False, num_workers=0)

dl = make()
it = iter(dl)
seen = [next(it)[0].flatten().tolist() for _ in range(3)]   # 3 of 6 batches
state = dl.state_dict()

dl2 = make()
dl2.load_state_dict(state)
resumed = [b[0].flatten().tolist() for b in dl2]

full = [b[0].flatten().tolist() for b in make()]
assert seen + resumed == full, (seen, resumed, full)
print(f"mid-epoch resume exact: {len(seen)} seen + {len(resumed)} resumed == {len(full)} full batches")